# Vecna / AIC51 — Chạy lại M09 & M10 trên Colab Pro+

**Cách dùng:** `Runtime → Change runtime type → A100 (hoặc L4) GPU + High-RAM`, rồi bấm **Run all**.
Chỉ có **một** thao tác thủ công: cấp quyền Google Drive ở Cell 2.

**Pipeline:** cài đặt → tải (aria2c) → giải nén + xoá zip → `aic51-cli add -d -k -a` → xoá video thô →
`analyse` (Qwen-VL → SigLIP2 → OCR → ASR) → kiểm tra toàn vẹn → nén **2 file ZIP** → upload Drive.

**Lưu ý khớp với `config.yaml` của repo:**
- OCR trong config dùng **Tesseract** (`source: "tesseract"`), nên notebook cài `tesseract-ocr-vie`. PaddleOCR/VietOCR
  **không** được pipeline gọi tới; bật `INSTALL_PADDLE_VIETOCR = True` ở Cell 1 nếu vẫn muốn cài.
- `api_key` Groq trong `config.yaml` sẽ bị **xoá trắng** khi copy sang workspace (LLM đang `enable: false`, không cần dùng trong bước này).
- Nếu model/pyannote cần token: thêm secret `HF_TOKEN` trong Colab (biểu tượng chìa khoá bên trái).
- Mỗi stage tự bỏ qua nếu đã hoàn tất (marker trong `/content/.state`) nên có thể Run all lại sau khi sửa lỗi.

## Cell 1 — Tham số

In [ ]:
# ===== THAM SỐ (chỉnh ở đây nếu cần) =====
REPO_URL       = "https://github.com/JimmyK300/Vecna.git"
REPO_BRANCH    = "main"
REPO_DIR       = "/content/Vecna"
WORKSPACE      = "/content/workspace"
DL_DIR         = "/content/downloads"        # nơi tải file .zip
RAW_DIR        = "/content/raw_videos"       # nơi giải nén video thô (tạm)
ZIP_DIR        = "/content/out_zip"          # nơi nén 2 file ZIP trước khi đẩy lên Drive
DRIVE_OUT_DIR  = "/content/drive/MyDrive/AIC2026/M09_M10"

BATCHES = {
    "M09": "https://aic-data.ledo.io.vn/Videos_M09.zip",
    "M10": "https://aic-data.ledo.io.vn/Videos_M10.zip",
}
ADD_FLAGS = "-d -k -a"                       # đúng như yêu cầu: -d -k -a

# Tên feature (chuẩn theo config.yaml) -> file .npy tương ứng cho mỗi frame
FEATURE_MODELS = ["qwen_vl", "image_siglip2_so400m-378", "ocr", "asr"]
EXPECTED_NPY   = [f"{m}.npy" for m in FEATURE_MODELS]

INSTALL_PADDLE_VIETOCR = True    # BẮT BUỘC: config.yaml của Vecna cấu hình OCR dùng paddle_vietocr (vgg_transformer)
PIN_COLAB_TORCH        = True    # ghim torch/torchvision/torchaudio của Colab để pip không thay bản CUDA
FLUSH_DRIVE_AT_END     = True    # đảm bảo Drive đồng bộ xong (flush_and_unmount) ở cuối
AUTO_DISCONNECT        = False   # True = tự ngắt runtime khi xong để tiết kiệm compute units
TZ = "Asia/Ho_Chi_Minh"

## Cell 2 — Logging, đo thời gian, tài nguyên & mount Google Drive

In [ ]:
import os, sys, re, time, shutil, codecs, subprocess, contextlib
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
import psutil

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

TZINFO         = ZoneInfo(TZ)
STATE_DIR      = Path("/content/.state"); STATE_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE       = "/content/pipeline_log.txt"
TORCH_CONSTRAINTS = "/content/torch_constraints.txt"
STAGE_TIMES    = []
PIPELINE_T0    = time.time()

def now():
    return datetime.now(TZINFO).strftime("%Y-%m-%d %H:%M:%S")

def log(msg=""):
    print(msg, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

def fmt_elapsed(sec):
    m, s = divmod(int(round(sec)), 60)
    return f"{m} phút {s} giây"

def gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20).stdout.strip().splitlines()
        parts = []
        for line in out:
            name, used, total = [x.strip() for x in line.split(",")]
            parts.append(f"{name} {float(used)/1024:.1f}/{float(total)/1024:.1f} GB")
        return " | ".join(parts) if parts else "N/A"
    except Exception:
        return "N/A"

def resources(tag):
    vm = psutil.virtual_memory()
    du = shutil.disk_usage("/content")
    log(f"    [{tag}] RAM dùng: {vm.used/2**30:.1f}/{vm.total/2**30:.1f} GB"
        f" | Đĩa trống: {du.free/2**30:.1f}/{du.total/2**30:.1f} GB"
        f" | VRAM: {gpu_status()}")

@contextlib.contextmanager
def timed(name, show_res=True):
    log("=" * 90)
    log(f"[{now()}] ▶ BẮT ĐẦU: {name}")
    if show_res: resources("trước")
    t0 = time.time()
    try:
        yield
    except BaseException as e:
        dt = time.time() - t0
        log(f"[{now()}] ✖ LỖI: {name} — {type(e).__name__}: {e}")
        log(f"    Elapsed time: {fmt_elapsed(dt)}")
        if show_res: resources("khi lỗi")
        STAGE_TIMES.append((name, dt, "LỖI"))
        raise
    dt = time.time() - t0
    log(f"[{now()}] ✔ KẾT THÚC: {name}")
    log(f"    Elapsed time: {fmt_elapsed(dt)}")
    if show_res: resources("sau")
    STAGE_TIMES.append((name, dt, "OK"))

def stage(name, key=None, heavy=True):
    # Decorator: CHẠY NGAY hàm được bọc, có đo thời gian; nếu có key và đã xong trước đó thì bỏ qua.
    def deco(fn):
        if key and (STATE_DIR / key).exists():
            log(f"[{now()}] ⏭  BỎ QUA (đã hoàn tất trước đó): {name}")
            return fn
        with timed(name, show_res=heavy):
            fn()
        if key:
            (STATE_DIR / key).touch()
        return fn
    return deco

def sh(cmd, cwd=None, check=True):
    # Chạy lệnh bash, stream output theo thời gian thực (giữ cả thanh tiến trình dùng \r), flush liên tục.
    log(f"$ {cmd}" + (f"   (cwd={cwd})" if cwd else ""))
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, env=dict(os.environ, DEBIAN_FRONTEND="noninteractive"))
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    fd = proc.stdout.fileno()
    while True:
        chunk = os.read(fd, 4096)
        if not chunk:
            break
        sys.stdout.write(dec.decode(chunk)); sys.stdout.flush()
    rc = proc.wait()
    if rc != 0 and check:
        raise RuntimeError(f"Lệnh thất bại (exit {rc}): {cmd}")
    return rc

def pip_install(args, cwd=None):
    base = f"pip install -q {args}"
    if PIN_COLAB_TORCH and Path(TORCH_CONSTRAINTS).exists():
        if sh(f"{base} -c {TORCH_CONSTRAINTS}", cwd=cwd, check=False) == 0:
            return
        log("⚠️  pip lỗi khi ghim torch → thử lại không ghim")
    sh(base, cwd=cwd)

# HF token (tuỳ chọn) từ Colab Secrets
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    log("HF_TOKEN: đã nạp từ Colab Secrets")
except Exception:
    log("HF_TOKEN: không có (bỏ qua — ổn nếu các model đều tải public)")

# Mount Google Drive (thao tác thủ công duy nhất)
@stage("Mount Google Drive", heavy=False)
def _():
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_OUT_DIR).mkdir(parents=True, exist_ok=True)
    log(f"    Thư mục đầu ra trên Drive: {DRIVE_OUT_DIR}")

## Cell 3 — Kiểm tra GPU / RAM

In [ ]:
@stage("Kiểm tra GPU / runtime", heavy=False)
def _():
    if sh("nvidia-smi", check=False) != 0:
        raise RuntimeError("Chưa bật GPU: Runtime → Change runtime type → A100/L4 GPU (+ High-RAM).")
    ram = psutil.virtual_memory().total / 2**30
    log(f"    RAM tổng: {ram:.1f} GB | CPU: {os.cpu_count()} core | Đĩa: {shutil.disk_usage('/content').total/2**30:.0f} GB")
    if ram < 40:
        log("⚠️  RAM < 40 GB — nên chọn High-RAM để analyse ổn định.")

## Cell 4 — Bước 1a: Cài đặt hệ thống (aria2, ffmpeg, unzip, zip, Tesseract-vie)

In [ ]:
@stage("Cài đặt gói hệ thống (apt)", key="apt")
def _():
    sh("apt-get update -qq")
    sh("apt-get install -y -qq aria2 ffmpeg unzip zip libsndfile1 "
       "tesseract-ocr tesseract-ocr-vie tesseract-ocr-eng")
    sh("aria2c --version | head -n 1; ffmpeg -version | head -n 1; tesseract --version 2>&1 | head -n 1", check=False)

## Cell 5 — Bước 1b: Clone repo & cài thư viện (thứ tự theo `note.md` của repo)

In [ ]:
@stage("Clone repo & cài đặt thư viện Python", key="pip")
def _():
    # 1) Clone (hoặc cập nhật) repo
    if Path(REPO_DIR, ".git").exists():
        sh(f"git -C {REPO_DIR} pull --ff-only", check=False)
    else:
        sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

    # 2) Ghim torch của Colab để các gói sau không kéo bản torch khác CUDA
    out = subprocess.run("pip list --format=freeze", shell=True, capture_output=True, text=True).stdout
    pins = [l for l in out.splitlines() if re.match(r"(?i)(torch|torchvision|torchaudio)==", l)]
    Path(TORCH_CONSTRAINTS).write_text("\n".join(pins) + "\n")
    log(f"    Ghim: {pins}")

    # 3) Thư viện mô hình (OpenCLIP/SigLIP2, Qwen-VL, OCR Tesseract)
    pip_install("-U open_clip_torch timm sentence-transformers transformers accelerate qwen-vl-utils "
                "pytesseract pyyaml psutil")
    if INSTALL_PADDLE_VIETOCR:
        # Thử cài paddlepaddle-gpu, nếu Colab xung đột CUDA thì fallback sang paddlepaddle
        if sh("pip install -q paddlepaddle-gpu paddleocr vietocr", check=False) != 0:
            log("⚠️  paddlepaddle-gpu lỗi phiên bản CUDA → Cài đặt paddlepaddle tiêu chuẩn...")
            sh("pip install -q paddlepaddle paddleocr vietocr")

    # 4) CLI nội bộ (cài SAU để ràng buộc phiên bản trong pyproject của repo được ưu tiên)
    pip_install("-e .", cwd=f"{REPO_DIR}/aic51-src")

    # 5) WhisperX theo đúng note.md: --no-deps rồi cài phụ thuộc thủ công
    pip_install("whisperx --no-deps")
    pip_install('faster-whisper "ctranslate2>=4.5.0" pyannote.audio nltk pandas')

### Kiểm tra nhanh môi trường sau cài đặt (fail sớm nếu thiếu thư viện)

In [ ]:
@stage("Kiểm tra import & CLI", heavy=False)
def _():
    check = r'''
import importlib
mods = ["torch","numpy","open_clip","timm","transformers","accelerate","sentence_transformers",
        "whisperx","faster_whisper","ctranslate2","pyannote.audio","paddleocr","vietocr"]
bad = []
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"  OK   {m:24s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        bad.append(m); print(f"  FAIL {m:24s} {type(e).__name__}: {e}")
import torch
print("  CUDA available:", torch.cuda.is_available(), "| torch", torch.__version__)
raise SystemExit(1 if (bad or not torch.cuda.is_available()) else 0)
'''
    Path("/content/_check_env.py").write_text(check)
    sh("python /content/_check_env.py")
    sh("aic51-cli --help")
    sh("tesseract --list-langs | grep -x vie")

## Cell 6 — Bước 1c: Khởi tạo workspace bằng `aic51-cli init` & đồng bộ `FINALCONFIG.yaml`


In [ ]:
FINAL_CONFIG_CONTENT = """# max workers ratio defining how much CPU cores to use
max_workers_ratio: 1.0

add:
  default_size: [1280, 720]
  # Maximum length between two consecutive keyframes (in seconds)
  max_scene_length: 2
  # Keyframe size ratio to the original frame. This also affects resolution of videos
  keyframe_resize_ratio: 1.0
  # Thumbnail size ratio to the keyframe
  thumbnail_resize_ratio: 0.25
  # Max clips length around the keyframes (in seconds)
  clip_length: 7
  # Video compress ratio
  compress_size_rate: 0.5

analyse:
  # num_workers of DataLoader
  num_workers: 4
  # pin_memory of DataLoader
  pin_memory: true

milvus:
  # Extra fields (apart from features)
  fields:
    - field_name: "frame_id"
      datatype: "VARCHAR"
      max_length: 32
      is_primary: true

# List of features
features: &analyse_features
  # image_clip_pe-l-14-336:
  #   model: "image_clip"
  #   source: "open_clip"
  #   arch_name: "PE-Core-L-14-336"
  #   pretrained_model: "meta"
  #   analyse:
  #     batch_size: 64
  #   index:
  #     datatype: "FLOAT_VECTOR"
  #     dim: 1024
  #     metric_type: "COSINE"
  #     index_type: "SCANN"
  #     params:
  #       nlist: 512

  image_siglip2_so400m-378:
    model: "image_siglip"
    source: "open_clip"
    arch_name: "ViT-SO400M-14-SigLIP2-378"
    pretrained_model: "webli"
    analyse:
      batch_size: 32
    index:
      datatype: "FLOAT_VECTOR"
      dim: 1152
      metric_type: "COSINE"
      index_type: "SCANN"
      params:
        nlist: 512

  qwen_vl:
    model: "qwen_vl_embedding"
    pretrained_model: "Qwen/Qwen3-VL-Embedding-2B"
    analyse:
      batch_size: 1
    index:
      datatype: "FLOAT_VECTOR"
      dim: 2048
      metric_type: "COSINE"
      index_type: "SCANN"
      params:
        nlist: 512

  ocr:
    model: "ocr"
    source: "paddle_vietocr"
    pretrained_model: "vgg_transformer"
    analyse:
      batch_size: 16
      pad_y: 6
      pad_x: 4
      det_lang: "vi"
    index:
      default_value: ""
      datatype: "VARCHAR"
      max_length: 8192
      index_type: "BM25"
      params:
        bm25_k1: 1.2
        bm25_b: 0.75
        inverted_index_algo: "TAAT_NAIVE"

  asr:
    model: "asr"
    source: "whisperx"
    arch_name: "large-v3-turbo"
    analyse:
      batch_size: 16
    index:
      default_value: ""
      datatype: "VARCHAR"
      max_length: 8192
      index_type: "BM25"
      params:
        bm25_k1: 1.2
        bm25_b: 0.75
        inverted_index_algo: "TAAT_NAIVE"

searcher:
  language_models:
    # language_clip_pe-l-14-336:
    #   model: "image_clip"
    #   source: "open_clip"
    #   arch_name: "PE-Core-L-14-336"
    #   pretrained_model: "meta"
    #   target:
    #     - image_clip_pe-l-14-336

    language_siglip2_so400m-378:
      model: "image_siglip"
      source: "open_clip"
      arch_name: "ViT-SO400M-14-SigLIP2-378"
      pretrained_model: "webli"
      target:
        - image_siglip2_so400m-378

    language_qwen_vl:
      model: "qwen_vl_embedding"
      pretrained_model: "Qwen/Qwen3-VL-Embedding-2B"
      target:
        - qwen_vl

  ocr:
    ocr_field: "ocr_sparse"

  asr:
    asr_field: "asr_sparse"

frontend:
  dev_port: 5173

backends:
  core:
    port: 6900
    search_proxy:
      request_timeout: null
      max_concurrent_requests: 10
      servers:
        - host: http://127.0.0.1:1337
    file_proxy:
      request_timeout: null
      max_concurrent_requests: 10
      servers:
        - host: http://127.0.0.1:4200

  search:
    port: 1337
    collection: "workspace_col"
    workers: 1
    gpu: true

  file:
    port: 4200
    workers: 1"""

@stage("Khởi tạo workspace & đồng bộ FINALCONFIG.yaml", key="init")
def _():
    Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
    if sh("aic51-cli init", cwd=WORKSPACE, check=False) != 0:
        log("    `aic51-cli init` (không đối số) thất bại → thử truyền đường dẫn workspace")
        sh(f"aic51-cli init {WORKSPACE}", cwd=WORKSPACE)

    # Ưu tiên lấy file FINALCONFIG.yaml từ repo nếu có, ngược lại ghi thẳng nội dung chuẩn
    repo_final_cfg = Path(REPO_DIR, "FINALCONFIG.yaml")
    dest_cfg = Path(WORKSPACE, "config.yaml")
    if repo_final_cfg.exists():
        shutil.copy(repo_final_cfg, dest_cfg)
        log("    [✓] Đã copy FINALCONFIG.yaml từ repo vào workspace/config.yaml")
    else:
        dest_cfg.write_text(FINAL_CONFIG_CONTENT, encoding="utf-8")
        log("    [✓] Đã ghi trực tiếp cấu hình chuẩn FINALCONFIG.yaml vào workspace/config.yaml")

    log("    Cấu trúc workspace:")
    for cur, dirs, files in os.walk(WORKSPACE):
        d = cur[len(WORKSPACE):].count(os.sep)
        if d >= 2: dirs[:] = []
        log("      " + "  " * d + os.path.basename(cur) + "/")


## Cell 7 — Bước 2a: Tải song song 2 file ZIP bằng `aria2c -x 16 -s 16`

In [ ]:
@stage("Tải video M09 & M10 (aria2c -x16 -s16)", key="download")
def _():
    import requests
    Path(DL_DIR).mkdir(parents=True, exist_ok=True)

    # Ước lượng dung lượng để cảnh báo sớm nếu đĩa không đủ (không bắt buộc)
    total = 0
    for tag, url in BATCHES.items():
        try:
            size = int(requests.head(url, allow_redirects=True, timeout=30).headers.get("Content-Length", 0))
            total += size
            log(f"    {tag}: {size/2**30:.2f} GB")
        except Exception as e:
            log(f"    {tag}: không lấy được dung lượng ({type(e).__name__})")
    free = shutil.disk_usage("/content").free
    if total and free < total * 2.1:
        log(f"⚠️  Đĩa trống {free/2**30:.0f} GB có thể không đủ (cần ~{total*2.1/2**30:.0f} GB cho zip + video giải nén).")

    lst = Path(DL_DIR, "urls.txt")
    lst.write_text("".join(f"{url}\n  out=Videos_{tag}.zip\n  dir={DL_DIR}\n" for tag, url in BATCHES.items()))
    sh(f"aria2c -x 16 -s 16 -j 2 -c --file-allocation=none --max-tries=10 --retry-wait=5 "
       f"--summary-interval=30 --console-log-level=warn -i {lst}")
    for tag in BATCHES:
        zp = Path(DL_DIR, f"Videos_{tag}.zip")
        assert zp.exists() and zp.stat().st_size > 0, f"Thiếu/rỗng: {zp}"
        log(f"    ✔ {zp.name}: {zp.stat().st_size/2**30:.2f} GB")

## Cell 8 — Bước 2b: Giải nén ra `/content/raw_videos/` và xoá `.zip` ngay

In [ ]:
VIDEO_EXT = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".ts", ".m4v"}

def flatten_videos(d):
    # Đưa mọi video nằm trong thư mục con ra thư mục gốc của batch (phòng zip có thư mục lồng nhau)
    d = Path(d); n = moved = 0
    for p in sorted(d.rglob("*")):
        if p.is_file() and p.suffix.lower() in VIDEO_EXT:
            n += 1
            if p.parent != d:
                target = d / p.name
                if target.exists():
                    log(f"⚠️  Trùng tên, bỏ qua: {p}"); continue
                shutil.move(str(p), str(target)); moved += 1
    for p in sorted(d.rglob("*"), reverse=True):
        if p.is_dir() and not any(p.iterdir()):
            p.rmdir()
    return n, moved

@stage("Giải nén video & xoá file .zip", key="extract")
def _():
    for tag in BATCHES:
        zp, out = Path(DL_DIR, f"Videos_{tag}.zip"), Path(RAW_DIR, tag)
        if zp.exists():
            out.mkdir(parents=True, exist_ok=True)
            with timed(f"unzip {zp.name}"):
                sh(f"unzip -q -o {zp} -d {out}")
            zp.unlink()  # xoá ngay để giải phóng đĩa
            log(f"    Đã xoá {zp.name}")
        elif not out.exists():
            raise FileNotFoundError(f"Không thấy {zp} lẫn {out}")
        n, moved = flatten_videos(out)
        log(f"    {tag}: {n} video (đã đưa {moved} file ra thư mục gốc)")
        if n == 0:
            raise RuntimeError(f"Không tìm thấy video nào trong {out}")

## Cell 9 — Bước 3: Trích xuất keyframes / thumbnails / audio bằng `aic51-cli add`

In [ ]:
@stage("Trích keyframes, thumbnails, audio (aic51-cli add)", key="add")
def _():
    for tag in BATCHES:
        with timed(f"aic51-cli add {tag} {ADD_FLAGS}"):
            sh(f"aic51-cli add {RAW_DIR}/{tag} {ADD_FLAGS}", cwd=WORKSPACE)
    for sub in ["keyframes", "thumbnails", "audio", "video_info"]:
        d = Path(WORKSPACE, "data", sub)
        n = sum(1 for p in d.rglob("*") if p.is_file()) if d.exists() else 0
        log(f"    data/{sub}: {n} file")

In [ ]:
@stage("Xoá video thô để tiết kiệm SSD", key="cleanup_raw", heavy=True)
def _():
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    log(f"    Đã xoá {RAW_DIR}")

## Cell 10 — Bước 4: Trích xuất đặc trưng (tuần tự, mỗi model một tiến trình → VRAM được giải phóng giữa các model)

In [ ]:
def analyse(model):
    sh(f"aic51-cli analyse -m {model}", cwd=WORKSPACE)

@stage("Feature 1/4 — Qwen-VL Embedding (qwen_vl)", key="analyse_qwen_vl")
def _(): analyse("qwen_vl")

In [ ]:
@stage("Feature 2/4 — SigLIP 2 (image_siglip2_so400m-378)", key="analyse_siglip2")
def _(): analyse("image_siglip2_so400m-378")

In [ ]:
@stage("Feature 3/4 — OCR (ocr)", key="analyse_ocr")
def _(): analyse("ocr")

In [ ]:
@stage("Feature 4/4 — ASR WhisperX (asr)", key="analyse_asr")
def _(): analyse("asr")

## Cell 11 — Bước 5: Kiểm tra toàn vẹn (4/4 file `.npy` mỗi frame & thumbnails khớp 1-1)

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def tree_preview(root, depth=2, limit=60):
    root, out = str(root), []
    for cur, dirs, files in os.walk(root):
        dirs.sort()
        d = cur[len(root):].count(os.sep)
        if d >= depth: dirs[:] = []
        out.append("  " * d + os.path.basename(cur) + "/")
        if len(out) >= limit: break
    return "\n".join(out)

def find_features_root():
    for c in ("features", "data/features"):
        p = Path(WORKSPACE, c)
        if p.is_dir() and next(p.rglob("*.npy"), None) is not None:
            return p
    raise FileNotFoundError("Không tìm thấy thư mục features/ chứa .npy trong workspace. Cấu trúc hiện có:\n"
                            + tree_preview(WORKSPACE, 3))

def norm_key(rel):
    return str(rel).replace(os.sep, "_")   # chịu được cả layout lồng (video/frame) lẫn phẳng (video_frame)

@stage("Kiểm tra toàn vẹn (verification)", heavy=False)
def _():
    root = find_features_root()
    thumbs_root = Path(WORKSPACE, "data", "thumbnails")
    log(f"    features root   : {root}")
    log(f"    thumbnails root : {thumbs_root}")

    # Gom các file .npy theo thư mục frame
    frames = {}
    for p in root.rglob("*.npy"):
        frames.setdefault(p.parent, {})[p.name] = p.stat().st_size
    if not any(set(EXPECTED_NPY) & set(v) for v in frames.values()):
        raise RuntimeError("Không thấy file nào tên qwen_vl.npy/... — layout features/ khác dự kiến. Cấu trúc:\n"
                           + tree_preview(root, 3))
    sample = next(iter(frames))
    log(f"    Ví dụ: {sample.relative_to(root)} -> {sorted(frames[sample])}")

    errors = []
    # (a) đủ 4/4 file, không rỗng
    incomplete = {}
    for d, files in frames.items():
        miss = [n for n in EXPECTED_NPY if n not in files]
        empty = [n for n in EXPECTED_NPY if files.get(n, 1) == 0]
        if miss or empty:
            incomplete[d] = (miss, empty)
    log(f"    Tổng số frame trong features/: {len(frames)}")
    for n in EXPECTED_NPY:
        log(f"      {n:34s}: {sum(1 for f in frames.values() if n in f)}/{len(frames)}")
    if incomplete:
        errors.append(f"{len(incomplete)} frame thiếu/rỗng file .npy")
        for d, (miss, empty) in list(incomplete.items())[:10]:
            log(f"      ✖ {d.relative_to(root)}: thiếu={miss} rỗng={empty}")

    # (b) cả M09 và M10 đều có frame
    for tag in BATCHES:
        n = sum(1 for d in frames if tag in str(d.relative_to(root)))
        log(f"    {tag}: {n} frame")
        if n == 0:
            errors.append(f"Không có frame nào của {tag} trong features/")

    # (c) thumbnails khớp 1-1 với folder frame
    if not thumbs_root.is_dir():
        raise FileNotFoundError(f"Không có {thumbs_root}")
    thumbs = {norm_key(p.relative_to(thumbs_root).with_suffix(""))
              for p in thumbs_root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT}
    keys = {norm_key(d.relative_to(root)) for d in frames}
    only_thumb, only_feat = sorted(thumbs - keys), sorted(keys - thumbs)
    log(f"    Thumbnails: {len(thumbs)} ảnh | Frame folder: {len(keys)}")
    if only_thumb or only_feat:
        errors.append(f"Thumbnails ≠ features: {len(only_thumb)} ảnh không có features, {len(only_feat)} frame không có thumbnail")
        log(f"      Chỉ có thumbnail (≤10): {only_thumb[:10]}")
        log(f"      Chỉ có features  (≤10): {only_feat[:10]}")

    if errors:
        raise AssertionError(" | ".join(errors))
    log(f"    ✔ {len(frames)} frame đủ 4/4 file và khớp 1-1 với {len(thumbs)} thumbnails")

## Cell 12 — Bước 6a: Nén đúng 2 file ZIP
`features_M09_M10.zip` và `thumbnails_M09_M10.zip`. **Không** nén `data/keyframes/`.
Đường dẫn bên trong zip giữ nguyên tương đối so với workspace, nên giải nén vào thư mục gốc workspace sẽ khôi phục đúng cấu trúc.

In [ ]:
@stage("Nén 2 file ZIP (features + thumbnails)")
def _():
    import zipfile
    out = Path(ZIP_DIR); out.mkdir(parents=True, exist_ok=True)
    feat_rel = str(find_features_root().relative_to(WORKSPACE))
    jobs = [("features_M09_M10.zip", feat_rel, 1),          # .npy nén kém → mức 1 cho nhanh
            ("thumbnails_M09_M10.zip", "data/thumbnails", 0)]  # ảnh đã nén sẵn → chỉ đóng gói
    for zname, rel, level in jobs:
        zp = out / zname
        zp.unlink(missing_ok=True)
        with timed(f"zip {zname} ← {rel}"):
            sh(f"zip -r -q -{level} {zp} {rel}", cwd=WORKSPACE)
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
        assert not any("keyframes" in n for n in names), "ZIP lỡ chứa keyframes!"
        log(f"    {zname}: {len(names)} mục, {zp.stat().st_size/2**30:.2f} GB")

## Cell 13 — Bước 6b: Upload lên Google Drive & tổng kết

In [ ]:
@stage("Upload 2 file ZIP lên Google Drive")
def _():
    dst = Path(DRIVE_OUT_DIR); dst.mkdir(parents=True, exist_ok=True)
    for zname in ["features_M09_M10.zip", "thumbnails_M09_M10.zip"]:
        src = Path(ZIP_DIR, zname)
        with timed(f"copy {zname} → Drive"):
            sh(f"cp {src} {dst}/")
        s1, s2 = src.stat().st_size, (dst / zname).stat().st_size
        assert s1 == s2, f"Kích thước lệch sau khi copy {zname}: local={s1} drive={s2}"
        log(f"    ✔ {dst/zname} ({s2/2**30:.2f} GB)")
    shutil.rmtree(ZIP_DIR, ignore_errors=True)  # giải phóng đĩa local

In [ ]:
log("=" * 90)
log("TỔNG KẾT THỜI GIAN CÁC BƯỚC")
for name, dt, st in STAGE_TIMES:
    log(f"  {st:4s} {fmt_elapsed(dt):>16s}   {name}")
log(f"[{now()}] TỔNG THỜI GIAN CHẠY: {fmt_elapsed(time.time() - PIPELINE_T0)}")
resources("cuối")

shutil.copy(LOG_FILE, Path(DRIVE_OUT_DIR, "pipeline_log.txt"))
log(f"Kết quả trên Drive: {DRIVE_OUT_DIR}")
for p in sorted(Path(DRIVE_OUT_DIR).iterdir()):
    log(f"  - {p.name}  ({p.stat().st_size/2**30:.2f} GB)")

if FLUSH_DRIVE_AT_END:
    from google.colab import drive
    drive.flush_and_unmount()
    log("Đã flush & unmount Drive (dữ liệu đã đồng bộ).")
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()